In [ ]:
import os
import glob
import numpy as np
import xarray as xr
import rioxarray
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchsummary import summary
import torch.optim as optim

from lib.dataset.dataloader import SICDataLoader


: 

In [ ]:
# Load data (update this path to your .nc directory)
sic_loader = SICDataLoader(
    data_dirs=['/dmidata/projects/asip-cms/tests/new_input_ncs/AMSR2/','/dmidata/projects/asip-cms/reproc/'],
    batch_size=1,
    shuffle=False,
    date_pattern=r'(\d{8})[T]\d{6}',
    year=2019,
)


In [ ]:
amsr2_file = sic_loader.get_matched_pairs_info()[-1]['amsr2_files'][0]
sic_file = sic_loader.get_matched_pairs_info()[-1]['sic_files'][0]
sic_file

In [ ]:
ds = xr.open_dataset(amsr2_file, engine='netcdf4')
ds

In [ ]:
ds = sic_loader._load_amsr2_frequencies(amsr2_file)
ds

In [ ]:
ds_sic = rioxarray.open_rasterio(sic_file, mask_and_scale=True)
ds_sic['band']

In [ ]:
import rioxarray
import cartopy.crs as ccrs
from rasterio.enums import Resampling


# 3. Handle the GCPs (The "Warp" Step)
# This is what converts the 'Identity' (0,0 to 5228, 1517) into real meters.
# We force it to EPSG:3411 to match your metadata.
ds_sic = ds_sic.rio.reproject(
    "EPSG:3411", 
    resampling=Resampling.bilinear
)

# 4. Clean up the dimensions
# open_rasterio often adds a 'band' dimension of length 1. 
# Squeezing makes it a 2D array (y, x), which is easier to work with.
# if 'band' in ds_sic.dims:
#     ds_sic = ds_sic.squeeze('band', drop=True)

# 5. Verify the fix
print(f"New CRS: {ds_sic.rio.crs}")
print(f"New Real-World Bounds: {ds_sic.rio.bounds()}")

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np

# 1. Define the Data Projection (Source)
# Based on your spatial_ref: Central Meridian -45, Latitude of Origin 70
data_crs = ccrs.Stereographic(
    central_latitude=90, 
    central_longitude=-45, 
    true_scale_latitude=70
)
plot_data = ds_sic.isel(band =1)
# 2. Setup the Figure
fig = plt.figure(figsize=(12, 10))
ax = plt.axes(projection=ccrs.NorthPolarStereo(central_longitude=-45))

# 3. Use your New Real-World Bounds to zoom in
# Format: [minx, maxx, miny, maxy]
# We add a 50km buffer so the edges aren't cut off
minx, miny, maxx, maxy = ds_sic.rio.bounds()
ax.set_extent([minx - 1000000, maxx + 1000000, miny - 1000000, maxy + 1000000], crs=data_crs)

# 4. Add Geography
ax.coastlines(resolution='10m', color='black', linewidth=1)
ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)

# 5. Plot the Data
# We use imshow because the grid is now regular after reprojecting
plot_data.plot.imshow(
    ax=ax,
    transform=data_crs,
    x='x', y='y',
    cmap='Blues_r',
    vmin=0, vmax=100, # Assuming SIC is 0-100%
    cbar_kwargs={'label': 'Sea Ice Concentration (%)'}
)
import cartopy.feature as cfeature

# Add this before plotting your data:
ax.add_feature(cfeature.LAND, facecolor='gray', zorder=0)

plt.title("Fram Strait / Greenland Sea - Sea Ice (EPSG:3411)")
plt.show()

In [ ]:
# 1. Select Band 1
band1 = ds_sic.isel(band=0)

# 2. Mask the 'No Data' value (255)
# .where() keeps data where the condition is True, and sets everything else to NaN
sic_data = band1.where(band1 != 254)

# 3. Plot with focused scale
fig = plt.figure(figsize=(12, 10))
ax = plt.axes(projection=ccrs.NorthPolarStereo(central_longitude=-45))

# Use the data_crs from before for the extent
ax.set_extent([minx, maxx, miny, maxy], crs=data_crs)
ax.coastlines(resolution='10m', color='black')

# Now the colorbar will represent 0% to 100%
sic_data.plot.imshow(
    ax=ax,
    transform=data_crs,
    x='x', y='y',
    cmap='viridis', # Dark blue for 100%, light for 0%
    vmin=0, vmax=100,
    cbar_kwargs={'label': 'Sea Ice Concentration (%)'}
)
import cartopy.feature as cfeature

# Add this before plotting your data:
# ax.add_feature(cfeature.LAND, facecolor='gray', zorder=0)

plt.title("Filtered Sea Ice Concentration (Masked 255)")
plt.show()

In [ ]:
# Take one sample
first_batch = next(iter(sic_loader))
sample = first_batch[0]

# Get first AMSR2 file and first available frequency channel
amsr2 = sample['input'][0]
freq_name = next(iter(amsr2['frequencies'].keys()))
freq_data = np.squeeze(amsr2['frequencies'][freq_name])


# Get first SIC file and first available SIC variable
sic = sample['ground_truth'][0]
sic_name = next(iter(sic['ground_truth'].keys()))
sic_data = np.squeeze(sic['ground_truth'][sic_name])



In [ ]:
print(sic['ground_truth']['band_2'])

In [ ]:
# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(amsr2['frequencies']['btemp_89.0v'], cmap='viridis')
axes[0].set_title('AMSR2')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(sic['ground_truth']['band_1'], cmap='gray')
axes[1].set_title(f"SIC ground truth: {sic_name}\nDate: {sample['date']}")
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()